#Consumir a API

In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Configuração e função de consumo da API
# MAGIC Carregado pelos outros notebooks com `%run ./00_config`.

import json
import os
import time
from datetime import datetime

import requests
from pyspark.sql import functions as F

CATALOG = "spotify"
RAW_PATH = f"/Volumes/spotify/landing/spotify_resource/origem"
MARKET = "BR"

# IDs dos artistas: copie da URL do perfil no Spotify
# ex.: https://open.spotify.com/artist/0TnOYISbd1XYRBk9myaseg -> 0TnOYISbd1XYRBk9myaseg
ARTIST_IDS = [
    "1w5Kfo2jwwIPruYS2UWh56",
    "58lV9VcRSjABbAbfWS6skp",
    "27T030eWyCQRmDyuvr1kxY",
    "3qm84nBOXUEQ2vnTfUTTFC"
]

# Token de acesso (Client Credentials). Vale por 1 hora.
client_id = dbutils.secrets.get("spotify", "spotify-client-id")
client_secret = dbutils.secrets.get("spotify", "spotify-client-secret")

resp = requests.post(
    "https://accounts.spotify.com/api/token",
    data={"grant_type": "client_credentials"},
    auth=(client_id, client_secret),
    timeout=30,
)
resp.raise_for_status()
headers = {"Authorization": f"Bearer {resp.json()['access_token']}"}

def consumir_api(endpoint, params=None):
    """Faz um GET na API do Spotify.
    endpoint: caminho relativo (ex.: 'artists/123') ou URL completa (ex.: o campo 'next' da paginação)."""
    if endpoint.startswith("http"):
        url = endpoint
    else:
        url = f"https://api.spotify.com/v1/{endpoint}"

    while True:
        resp = requests.get(url, headers=headers, params=params, timeout=30)

        # Rate limit: espera o tempo indicado pela API e tenta de novo
        if resp.status_code == 429:
            espera = int(resp.headers.get("Retry-After", 5))
            print(f"Rate limit, aguardando {espera}s...")
            time.sleep(espera)
            continue

        resp.raise_for_status()
        return resp.json()

#Camada Bronze

In [0]:
# Bronze Artist
artistas = []

for artist_id in ARTIST_IDS:
    artista = consumir_api(f"artists/{artist_id}")
    artistas.append(artista)
    print("OK:", artista["name"])

# Salvar o JSON bruto no volume (um objeto por linha)
data_carga = datetime.now().strftime("%Y-%m-%d_%H%M%S")
caminho = f"{RAW_PATH}/artists/artists_{data_carga}.json"

try:    os.makedirs(os.path.dirname(caminho), exist_ok=True)
except OSError as e:
    print(f"Could not create directory: {e}")

with open(caminho, "w", encoding="utf-8") as f:
    for artista in artistas:
        f.write(json.dumps(artista, ensure_ascii=False) + "\n")

print(f"{len(artistas)} artistas salvos em {caminho}")

# Gravar na bronze, sem transformação, só com colunas de controle
df = (
    spark.read.json(caminho)
    .withColumn("_ingestion_timestamp", F.current_timestamp())
    .withColumn("_source_file", F.lit(caminho))
)

(
    df.write
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(f"{CATALOG}.bronze.artists")
)

display(df)

In [0]:
# Album Tracks

album_ids = [
    linha["id"]
    for linha in spark.table(f"{CATALOG}.bronze.albums").select("id").distinct().collect()
]

print(f"{len(album_ids)} álbuns para buscar")

# Consumir a API, percorrendo todas as páginas de cada álbum
faixas = []

for album_id in album_ids:
    endpoint = f"albums/{album_id}/tracks"
    params = {"market": MARKET, "limit": 20}

    while endpoint:
        resposta = consumir_api(endpoint, params)

        for faixa in resposta["items"]:
            faixa["album_id"] = album_id  # a resposta não traz o álbum, então guardamos aqui
            faixas.append(faixa)

        endpoint = resposta["next"]
        params = None

print(f"Total de faixas: {len(faixas)}")

# COMMAND ----------

# Salvar o JSON bruto no volume (um objeto por linha)
data_carga = datetime.now().strftime("%Y-%m-%d_%H%M%S")
caminho = f"{RAW_PATH}/album_tracks/album_tracks_{data_carga}.json"

os.makedirs(os.path.dirname(caminho), exist_ok=True)
with open(caminho, "w", encoding="utf-8") as f:
    for faixa in faixas:
        f.write(json.dumps(faixa, ensure_ascii=False) + "\n")

print(f"{len(faixas)} faixas salvas em {caminho}")

# Gravar na bronze, sem transformação, só com colunas de controle
df = (
    spark.read.json(caminho)
    .withColumn("_ingestion_timestamp", F.current_timestamp())
    .withColumn("_source_file", F.lit(caminho))
)

(
    df.write
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(f"{CATALOG}.bronze.album_tracks")
)

display(df)

In [0]:
# Albuns

albuns = []

for artist_id in ARTIST_IDS:
    endpoint = f"artists/{artist_id}/albums"
    params = {"include_groups": "album,single", "market": MARKET, "limit": 10}

    while endpoint:
        resposta = consumir_api(endpoint, params)

        for album in resposta["items"]:
            album["artist_id"] = artist_id  # guarda de qual artista veio a busca
            albuns.append(album)

        endpoint = resposta["next"]  # URL da próxima página, ou None na última
        params = None                # a URL "next" já traz os parâmetros

    print(f"OK: {artist_id}")

print(f"Total de álbuns: {len(albuns)}")

# Salvar o JSON bruto no volume (um objeto por linha)
data_carga = datetime.now().strftime("%Y-%m-%d_%H%M%S")
caminho = f"{RAW_PATH}/albums/albums_{data_carga}.json"

os.makedirs(os.path.dirname(caminho), exist_ok=True)
with open(caminho, "w", encoding="utf-8") as f:
    for album in albuns:
        f.write(json.dumps(album, ensure_ascii=False) + "\n")

print(f"{len(albuns)} álbuns salvos em {caminho}")

# Gravar na bronze, sem transformação, só com colunas de controle
df = (
    spark.read.json(caminho)
    .withColumn("_ingestion_timestamp", F.current_timestamp())
    .withColumn("_source_file", F.lit(caminho))
)

(
    df.write
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(f"{CATALOG}.bronze.albums")
)

display(df)